# Reproduction notebook: 18_weather_electricity_official_patchtst_baselines

This notebook is retained as an executable provenance record for the anonymous supplementary package. Saved outputs and internal development notes have been removed.


In [ ]:

from pathlib import Path
from types import SimpleNamespace
import gc
import importlib
import json
import os
import random
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 420)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATASETS = ["Weather", "Electricity"]
HORIZONS = [96, 192, 336, 720]

SEQ_LEN = 336
LABEL_LEN = 48
SEED = 2021

RESULT_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "weather_electricity_official_patchtst_baselines"
)

CHECKPOINT_DIR = RESULT_ROOT / "full_direct"
HISTORY_DIR = RESULT_ROOT / "history"
ARTIFACT_DIR = RESULT_ROOT / "artifacts"

for p in [
    RESULT_ROOT,
    CHECKPOINT_DIR,
    HISTORY_DIR,
    ARTIFACT_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

RESUME = True
FORCE_RETRAIN = False

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Output:", RESULT_ROOT)


In [ ]:

OFFICIAL_REPO_CANDIDATES = [
    Path(
        "/code/stock_regime_retrieval/"
        "strong_forecaster/PatchTST_official"
    ),
    Path("/code/PatchTST_official"),
    Path("/data/PatchTST_official"),
    Path("/data/PatchTST"),
]

OFFICIAL_REPO = next(
    (
        p for p in OFFICIAL_REPO_CANDIDATES
        if (
            p
            / "PatchTST_supervised"
            / "models"
            / "PatchTST.py"
        ).exists()
    ),
    None,
)

if OFFICIAL_REPO is None:
    OFFICIAL_REPO = OFFICIAL_REPO_CANDIDATES[0]
    OFFICIAL_REPO.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "Official repository not found. Cloning:",
        OFFICIAL_REPO,
    )

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/yuqinie98/PatchTST.git",
            str(OFFICIAL_REPO),
        ],
        check=True,
    )

SUPERVISED_ROOT = (
    OFFICIAL_REPO
    / "PatchTST_supervised"
)

if not (
    SUPERVISED_ROOT
    / "models"
    / "PatchTST.py"
).exists():
    raise FileNotFoundError(
        f"Official supervised model not found: {SUPERVISED_ROOT}"
    )

try:
    commit = subprocess.check_output(
        [
            "git",
            "-C",
            str(OFFICIAL_REPO),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    ).strip()
except Exception:
    commit = "unknown"

print("Official repo:", OFFICIAL_REPO)
print("Supervised root:", SUPERVISED_ROOT)
print("Git commit:", commit)

(ARTIFACT_DIR / "official_repo_commit.txt").write_text(
    commit + "\n"
)


In [ ]:

# Avoid accidental import of Time-Series-Library modules
# with the same top-level names.
for module_name in list(sys.modules.keys()):
    if (
        module_name == "models"
        or module_name.startswith("models.")
        or module_name == "layers"
        or module_name.startswith("layers.")
    ):
        del sys.modules[module_name]

if str(SUPERVISED_ROOT) in sys.path:
    sys.path.remove(str(SUPERVISED_ROOT))

sys.path.insert(0, str(SUPERVISED_ROOT))

patchtst_module = importlib.import_module(
    "models.PatchTST"
)

OfficialPatchTST = patchtst_module.Model

actual_model_file = Path(
    patchtst_module.__file__
).resolve()

expected_model_file = (
    SUPERVISED_ROOT
    / "models"
    / "PatchTST.py"
).resolve()

print("Imported model:", actual_model_file)

if actual_model_file != expected_model_file:
    raise RuntimeError(
        "Wrong PatchTST implementation imported.\n"
        f"Expected: {expected_model_file}\n"
        f"Actual:   {actual_model_file}"
    )

print(
    "PASS: official PatchTST supervised implementation is active."
)


In [ ]:

DATA_PATH_CANDIDATES = {
    "Weather": [
        Path("/data/dataset/weather.csv"),
        Path("/data/dataset/weather/weather.csv"),
        Path("/data/Time-Series-Library/dataset/weather/weather.csv"),
        Path("/data/Time-Series-Library/dataset/weather.csv"),
        Path("/data/Time-Series-Library_v2/dataset/weather/weather.csv"),
        Path("/data/Time-Series-Library_v2/dataset/weather.csv"),
        Path("/code/stock_regime_retrieval/strong_forecaster/data/weather.csv"),
    ],
    "Electricity": [
        Path("/data/dataset/electricity.csv"),
        Path("/data/dataset/electricity/electricity.csv"),
        Path("/data/Time-Series-Library/dataset/electricity/electricity.csv"),
        Path("/data/Time-Series-Library/dataset/electricity.csv"),
        Path("/data/Time-Series-Library_v2/dataset/electricity/electricity.csv"),
        Path("/data/Time-Series-Library_v2/dataset/electricity.csv"),
        Path("/code/stock_regime_retrieval/strong_forecaster/data/electricity.csv"),
    ],
}

DATA_PATHS = {}

for name, candidates in DATA_PATH_CANDIDATES.items():
    hit = next(
        (p for p in candidates if p.is_file()),
        None,
    )

    DATA_PATHS[name] = hit

    print(
        f"{name:11s}:",
        hit if hit is not None else "NOT FOUND",
    )

missing = [
    name
    for name, path in DATA_PATHS.items()
    if path is None
]

if missing:
    print("\nAttempted paths:")
    for name in missing:
        print(f"\n{name}")
        for p in DATA_PATH_CANDIDATES[name]:
            print(" -", p)

    raise FileNotFoundError(
        "Dataset file(s) not found: "
        + ", ".join(missing)
        + ". Add the actual path to DATA_PATH_CANDIDATES."
    )


In [ ]:

EXPECTED_CHANNELS = {
    "Weather": 21,
    "Electricity": 321,
}


def prepare_custom_data(name):
    path = DATA_PATHS[name]
    df_raw = pd.read_csv(path)

    if "date" not in df_raw.columns:
        raise ValueError(
            f"{name}: official Dataset_Custom requires a 'date' column."
        )

    # Official Dataset_Custom moves target 'OT' to the final column.
    # For M forecasting this preserves the exact official column ordering.
    if "OT" not in df_raw.columns:
        raise ValueError(
            f"{name}: official script uses the default target='OT', "
            "but the CSV has no OT column. "
            f"First columns: {df_raw.columns.tolist()[:10]}"
        )

    cols = list(df_raw.columns)
    cols.remove("OT")
    cols.remove("date")

    ordered = df_raw[
        ["date"] + cols + ["OT"]
    ].copy()

    value_cols = list(
        ordered.columns[1:]
    )

    raw = ordered[
        value_cols
    ].to_numpy(
        dtype=np.float32
    )

    n = len(raw)

    num_train = int(n * 0.7)
    num_test = int(n * 0.2)
    num_val = n - num_train - num_test

    train_end = num_train
    val_end = num_train + num_val
    test_end = n

    scaler = StandardScaler()
    scaler.fit(
        raw[:train_end]
    )

    z = scaler.transform(
        raw
    ).astype(
        np.float32
    )

    expected_c = EXPECTED_CHANNELS[name]

    if raw.shape[1] != expected_c:
        raise ValueError(
            f"{name}: expected {expected_c} channels from the official "
            f"script but found {raw.shape[1]}."
        )

    return {
        "name": name,
        "path": path,
        "df": ordered,
        "columns": value_cols,
        "raw": raw,
        "z": z,
        "mean": scaler.mean_.astype(np.float32),
        "scale": scaler.scale_.astype(np.float32),
        "n_channels": raw.shape[1],
        "n": n,
        "train_end": train_end,
        "val_end": val_end,
        "test_end": test_end,
        "num_train": num_train,
        "num_val": num_val,
        "num_test": num_test,
    }


DATA = {
    name: prepare_custom_data(name)
    for name in DATASETS
}

protocol_df = pd.DataFrame([
    {
        "Dataset": name,
        "Path": str(d["path"]),
        "Rows": d["n"],
        "Channels": d["n_channels"],
        "TrainRows": d["num_train"],
        "ValRows": d["num_val"],
        "TestRows": d["num_test"],
        "TrainEnd": d["train_end"],
        "ValEnd": d["val_end"],
        "TestEnd": d["test_end"],
    }
    for name, d in DATA.items()
])

display(protocol_df)

protocol_df.to_csv(
    ARTIFACT_DIR / "dataset_protocol.csv",
    index=False,
)


In [ ]:

RECIPES = {
    "Weather": {
        "enc_in": 21,
        "e_layers": 3,
        "n_heads": 16,
        "d_model": 128,
        "d_ff": 256,
        "dropout": 0.2,
        "fc_dropout": 0.2,
        "head_dropout": 0.0,
        "patch_len": 16,
        "stride": 8,
        "batch_size": 128,
        "train_epochs": 100,
        "patience": 20,
        "learning_rate": 1e-4,
        # weather.sh does not override these parser defaults.
        "lradj": "type3",
        "pct_start": 0.3,
        "seed": 2021,
    },
    "Electricity": {
        "enc_in": 321,
        "e_layers": 3,
        "n_heads": 16,
        "d_model": 128,
        "d_ff": 256,
        "dropout": 0.2,
        "fc_dropout": 0.2,
        "head_dropout": 0.0,
        "patch_len": 16,
        "stride": 8,
        "batch_size": 32,
        "train_epochs": 100,
        "patience": 10,
        "learning_rate": 1e-4,
        "lradj": "TST",
        "pct_start": 0.2,
        "seed": 2021,
    },
}

display(
    pd.DataFrame(RECIPES).T
)


In [ ]:

class CustomForecastDataset(Dataset):
    def __init__(
        self,
        name,
        flag,
        horizon,
    ):
        super().__init__()

        if flag not in {
            "train",
            "val",
            "test",
        }:
            raise ValueError(flag)

        self.name = name
        self.flag = flag
        self.horizon = int(horizon)

        d = DATA[name]
        z = d["z"]

        border1s = [
            0,
            d["train_end"] - SEQ_LEN,
            d["val_end"] - SEQ_LEN,
        ]

        border2s = [
            d["train_end"],
            d["val_end"],
            d["test_end"],
        ]

        idx = {
            "train": 0,
            "val": 1,
            "test": 2,
        }[flag]

        self.border1 = int(border1s[idx])
        self.border2 = int(border2s[idx])

        self.data_x = z[
            self.border1:
            self.border2
        ]

    def __getitem__(self, index):
        s_begin = index
        s_end = s_begin + SEQ_LEN

        r_begin = s_end - LABEL_LEN
        r_end = (
            r_begin
            + LABEL_LEN
            + self.horizon
        )

        seq_x = self.data_x[
            s_begin:s_end
        ]

        seq_y = self.data_x[
            r_begin:r_end
        ]

        return (
            torch.from_numpy(seq_x),
            torch.from_numpy(seq_y),
        )

    def __len__(self):
        return (
            len(self.data_x)
            - SEQ_LEN
            - self.horizon
            + 1
        )


window_rows = []

for name in DATASETS:
    for h in HORIZONS:
        for flag in [
            "train",
            "val",
            "test",
        ]:
            ds = CustomForecastDataset(
                name,
                flag,
                h,
            )

            window_rows.append({
                "Dataset": name,
                "Horizon": h,
                "Split": flag,
                "Windows": len(ds),
                "Border1": ds.border1,
                "Border2": ds.border2,
            })

display(
    pd.DataFrame(window_rows)
)


In [ ]:

def make_config(name, horizon):
    r = RECIPES[name]

    return SimpleNamespace(
        enc_in=r["enc_in"],
        seq_len=SEQ_LEN,
        pred_len=int(horizon),
        e_layers=r["e_layers"],
        n_heads=r["n_heads"],
        d_model=r["d_model"],
        d_ff=r["d_ff"],
        dropout=r["dropout"],
        fc_dropout=r["fc_dropout"],
        head_dropout=r["head_dropout"],
        individual=0,
        patch_len=r["patch_len"],
        stride=r["stride"],
        padding_patch="end",
        revin=1,
        affine=0,
        subtract_last=0,
        decomposition=0,
        kernel_size=25,
    )


def build_model(name, horizon):
    model = OfficialPatchTST(
        make_config(
            name,
            horizon,
        )
    ).float().to(DEVICE)

    return model


param_rows = []

for name in DATASETS:
    for h in HORIZONS:
        model = build_model(name, h)

        params = sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )

        param_rows.append({
            "Dataset": name,
            "Horizon": h,
            "TrainableParams": params,
            "TrainableParams_M": params / 1e6,
        })

        del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

display(
    pd.DataFrame(param_rows)
)


In [ ]:

def make_loaders(name, horizon):
    r = RECIPES[name]

    train_ds = CustomForecastDataset(
        name,
        "train",
        horizon,
    )

    val_ds = CustomForecastDataset(
        name,
        "val",
        horizon,
    )

    test_ds = CustomForecastDataset(
        name,
        "test",
        horizon,
    )

    # Official data_factory:
    # train/val: shuffle=True, drop_last=True
    # test: shuffle=False, drop_last=True
    train_loader = DataLoader(
        train_ds,
        batch_size=r["batch_size"],
        shuffle=True,
        num_workers=0,
        drop_last=True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=r["batch_size"],
        shuffle=True,
        num_workers=0,
        drop_last=True,
    )

    official_test_loader = DataLoader(
        test_ds,
        batch_size=r["batch_size"],
        shuffle=False,
        num_workers=0,
        drop_last=True,
    )

    # Our matched evaluation for Experiment 19.
    full_test_loader = DataLoader(
        test_ds,
        batch_size=r["batch_size"],
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )

    return {
        "train_ds": train_ds,
        "val_ds": val_ds,
        "test_ds": test_ds,
        "train": train_loader,
        "val": val_loader,
        "official_test": official_test_loader,
        "full_test": full_test_loader,
    }


In [ ]:

@torch.no_grad()
def evaluate_loader(
    model,
    loader,
    horizon,
):
    model.eval()

    batch_mse = []

    sse = 0.0
    sae = 0.0
    count = 0
    windows = 0

    for batch_x, batch_y in loader:
        x = batch_x.float().to(DEVICE)

        y = batch_y[
            :,
            -horizon:,
            :
        ].float().to(DEVICE)

        pred = model(x)[
            :,
            -horizon:,
            :
        ]

        err = pred - y

        batch_mse.append(
            float(
                (err ** 2).mean().item()
            )
        )

        sse += float(
            (err ** 2).sum().item()
        )

        sae += float(
            err.abs().sum().item()
        )

        count += err.numel()
        windows += len(x)

        del x, y, pred, err

    return {
        "BatchAverageMSE": float(
            np.mean(batch_mse)
        ),
        "MSE": sse / count,
        "MAE": sae / count,
        "Elements": count,
        "Batches": len(batch_mse),
        "Windows": windows,
    }


In [ ]:

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Keep behavior close to the official script.
    torch.backends.cudnn.benchmark = False


def adjust_type3_lr(
    optimizer,
    base_lr,
    epoch,
):
    # Official PatchTST utils/tools.py type3 rule.
    lr = (
        base_lr
        if epoch < 3
        else base_lr
        * (
            0.9
            ** (
                epoch - 3
            )
        )
    )

    for group in optimizer.param_groups:
        group["lr"] = lr

    return lr


In [ ]:

def checkpoint_path(
    name,
    horizon,
):
    safe = name.lower()

    return (
        CHECKPOINT_DIR
        / (
            f"{safe}_L336_H{horizon}_"
            "official_recipe_seed2021.pt"
        )
    )


def history_path(
    name,
    horizon,
):
    safe = name.lower()

    return (
        HISTORY_DIR
        / (
            f"{safe}_L336_H{horizon}_"
            "official_recipe_seed2021.csv"
        )
    )


In [ ]:

def train_or_load(
    name,
    horizon,
):
    r = RECIPES[name]
    ckpt_path = checkpoint_path(
        name,
        horizon,
    )

    if (
        RESUME
        and ckpt_path.exists()
        and not FORCE_RETRAIN
    ):
        ckpt = torch.load(
            ckpt_path,
            map_location=DEVICE,
        )

        model = build_model(
            name,
            horizon,
        )

        model.load_state_dict(
            ckpt["StateDict"]
        )

        model.eval()

        print(
            f"Loaded {name} H={horizon}: "
            f"best={ckpt['BestValMSE']:.6f}"
            f"@{ckpt['BestEpoch']}"
        )

        return (
            model,
            ckpt,
            make_loaders(
                name,
                horizon,
            ),
        )

    set_seed(
        r["seed"]
    )

    loaders = make_loaders(
        name,
        horizon,
    )

    model = build_model(
        name,
        horizon,
    )

    # Official exp_main.py uses Adam.
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=r["learning_rate"],
    )

    onecycle = torch.optim.lr_scheduler.OneCycleLR(
        optimizer=optimizer,
        steps_per_epoch=len(
            loaders["train"]
        ),
        pct_start=r["pct_start"],
        epochs=r["train_epochs"],
        max_lr=r["learning_rate"],
    )

    best_val = float("inf")
    best_epoch = -1
    best_state = None
    wait = 0
    history = []

    for epoch in range(
        1,
        r["train_epochs"] + 1,
    ):
        model.train()

        losses = []
        t0 = time.time()

        for batch_x, batch_y in loaders["train"]:
            x = batch_x.float().to(DEVICE)

            y = batch_y[
                :,
                -horizon:,
                :
            ].float().to(DEVICE)

            optimizer.zero_grad(
                set_to_none=True
            )

            pred = model(x)[
                :,
                -horizon:,
                :
            ]

            loss = F.mse_loss(
                pred,
                y,
            )

            loss.backward()
            optimizer.step()

            losses.append(
                float(
                    loss.item()
                )
            )

            if r["lradj"] == "TST":
                # Match official PatchTST ordering:
                # read current OneCycle LR, assign it,
                # then advance scheduler.
                current = onecycle.get_last_lr()[0]

                for group in optimizer.param_groups:
                    group["lr"] = current

                onecycle.step()

            del x, y, pred, loss

        train_mse = float(
            np.mean(losses)
        )

        val = evaluate_loader(
            model,
            loaders["val"],
            horizon,
        )

        # Official EarlyStopping receives mean validation batch MSE.
        val_for_selection = val[
            "BatchAverageMSE"
        ]

        if (
            val_for_selection
            < best_val
            - 1e-12
        ):
            best_val = val_for_selection
            best_epoch = epoch

            best_state = {
                k:
                    v.detach()
                    .cpu()
                    .clone()
                for k, v
                in model.state_dict().items()
            }

            wait = 0
        else:
            wait += 1

        if r["lradj"] != "TST":
            lr = adjust_type3_lr(
                optimizer,
                r["learning_rate"],
                epoch,
            )
        else:
            lr = onecycle.get_last_lr()[0]

        history.append({
            "Epoch": epoch,
            "TrainMSE": train_mse,
            "ValBatchAverageMSE": val_for_selection,
            "ValGlobalMSE": val["MSE"],
            "ValMAE": val["MAE"],
            "BestValMSE": best_val,
            "BestEpoch": best_epoch,
            "LR": lr,
            "Seconds": time.time() - t0,
        })

        print(
            f"{name:11s} H={horizon:3d} "
            f"ep={epoch:03d}/{r['train_epochs']} | "
            f"train={train_mse:.6f} | "
            f"val={val_for_selection:.6f} | "
            f"best={best_val:.6f}@{best_epoch} | "
            f"lr={lr:.3e} | "
            f"wait={wait}/{r['patience']}"
        )

        pd.DataFrame(
            history
        ).to_csv(
            history_path(
                name,
                horizon,
            ),
            index=False,
        )

        if wait >= r["patience"]:
            print(
                f"Early stopping: {name} H={horizon} "
                f"at epoch {epoch}."
            )
            break

    if best_state is None:
        raise RuntimeError(
            f"{name} H={horizon}: no checkpoint selected."
        )

    model.load_state_dict(
        best_state
    )

    model.eval()

    ckpt = {
        "Dataset": name,
        "SeqLen": SEQ_LEN,
        "PredLen": int(horizon),
        "Seed": r["seed"],
        "Recipe": r,
        "OfficialRepoCommit": commit,
        "BestEpoch": int(best_epoch),
        "BestValMSE": float(best_val),
        "StateDict": best_state,
    }

    torch.save(
        ckpt,
        ckpt_path,
    )

    return (
        model,
        ckpt,
        loaders,
    )


In [ ]:

SUMMARY_PATH = (
    RESULT_ROOT
    / "summary.csv"
)

existing = (
    pd.read_csv(
        SUMMARY_PATH
    )
    if (
        RESUME
        and SUMMARY_PATH.exists()
    )
    else pd.DataFrame()
)

result_rows = (
    existing.to_dict("records")
    if len(existing)
    else []
)


def already_done(
    name,
    horizon,
):
    if not len(existing):
        return False

    return bool(
        (
            (existing["Dataset"] == name)
            & (
                existing["Horizon"]
                == horizon
            )
        ).any()
    )


for name in DATASETS:
    for horizon in HORIZONS:
        if already_done(
            name,
            horizon,
        ):
            print(
                f"SKIP completed: {name} H={horizon}"
            )
            continue

        print(
            "\n"
            + "=" * 140
        )

        print(
            f"OFFICIAL PATCHTST | "
            f"{name} | L=336 | H={horizon}"
        )

        print(
            "=" * 140
        )

        t0 = time.time()

        model, ckpt, loaders = train_or_load(
            name,
            horizon,
        )

        official_style = evaluate_loader(
            model,
            loaders["official_test"],
            horizon,
        )

        full_stride1 = evaluate_loader(
            model,
            loaders["full_test"],
            horizon,
        )

        row = {
            "Dataset": name,
            "Horizon": horizon,
            "SeqLen": SEQ_LEN,
            "Channels": DATA[name]["n_channels"],
            "BestEpoch": ckpt["BestEpoch"],
            "BestValMSE": ckpt["BestValMSE"],
            "OfficialStyle_MSE": official_style["MSE"],
            "OfficialStyle_MAE": official_style["MAE"],
            "OfficialStyle_BatchAverageMSE": (
                official_style["BatchAverageMSE"]
            ),
            "OfficialStyle_Windows": (
                official_style["Windows"]
            ),
            "FullStride1_MSE": full_stride1["MSE"],
            "FullStride1_MAE": full_stride1["MAE"],
            "FullStride1_Windows": (
                full_stride1["Windows"]
            ),
            "ExcludedByOfficialDropLast": (
                full_stride1["Windows"]
                - official_style["Windows"]
            ),
            "RuntimeMinutes": (
                time.time() - t0
            ) / 60.0,
            "Checkpoint": str(
                checkpoint_path(
                    name,
                    horizon,
                )
            ),
            "OfficialRepoCommit": commit,
        }

        result_rows.append(row)

        pd.DataFrame(
            result_rows
        ).to_csv(
            SUMMARY_PATH,
            index=False,
        )

        display(
            pd.DataFrame([row])[
                [
                    "Dataset",
                    "Horizon",
                    "BestEpoch",
                    "BestValMSE",
                    "OfficialStyle_MSE",
                    "OfficialStyle_MAE",
                    "FullStride1_MSE",
                    "FullStride1_MAE",
                    "FullStride1_Windows",
                    "ExcludedByOfficialDropLast",
                ]
            ]
        )

        del (
            model,
            ckpt,
            loaders,
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


summary_df = (
    pd.DataFrame(result_rows)
    .sort_values(
        [
            "Dataset",
            "Horizon",
        ]
    )
    .reset_index(drop=True)
)

display(summary_df)


In [ ]:

compact = summary_df[
    [
        "Dataset",
        "Horizon",
        "OfficialStyle_MSE",
        "OfficialStyle_MAE",
        "FullStride1_MSE",
        "FullStride1_MAE",
        "BestEpoch",
        "BestValMSE",
    ]
].copy()

display(compact)

compact.to_csv(
    RESULT_ROOT
    / "compact_baselines.csv",
    index=False,
)


In [ ]:

OLD_SUMMARY_CANDIDATES = [
    Path(
        "/data/dataset/strong_forecaster/"
        "multidataset_crossfit_screening/"
        "summary.csv"
    ),
    Path(
        "/data/dataset/strong_forecaster/"
        "multidataset_crossfit_screening/"
        "screening_summary.csv"
    ),
]

old_path = next(
    (
        p
        for p in OLD_SUMMARY_CANDIDATES
        if p.is_file()
    ),
    None,
)

if old_path is None:
    print(
        "Old Experiment 13 summary not found. "
        "This comparison is optional."
    )
else:
    old = pd.read_csv(old_path)

    print(
        "Old screening summary:",
        old_path,
    )

    # Handle the column names used in our previous notebooks.
    direct_col = next(
        (
            c
            for c in [
                "PatchTST_MSE",
                "Direct_MSE",
                "DirectMSE",
            ]
            if c in old.columns
        ),
        None,
    )

    horizon_col = next(
        (
            c
            for c in [
                "Horizon",
                "PredLen",
                "pred_len",
            ]
            if c in old.columns
        ),
        None,
    )

    dataset_col = next(
        (
            c
            for c in [
                "Dataset",
                "dataset",
            ]
            if c in old.columns
        ),
        None,
    )

    if (
        direct_col is None
        or horizon_col is None
        or dataset_col is None
    ):
        print(
            "Could not identify the old direct-baseline columns. "
            "Available columns:"
        )
        print(
            old.columns.tolist()
        )
    else:
        old_sub = old[
            old[
                dataset_col
            ].isin(
                DATASETS
            )
        ][
            [
                dataset_col,
                horizon_col,
                direct_col,
            ]
        ].copy()

        old_sub.columns = [
            "Dataset",
            "Horizon",
            "OldScreening_MSE",
        ]

        comparison = compact.merge(
            old_sub,
            on=[
                "Dataset",
                "Horizon",
            ],
            how="left",
        )

        comparison[
            "StrongGainVsOld_pct"
        ] = (
            100.0
            * (
                comparison[
                    "OldScreening_MSE"
                ]
                - comparison[
                    "FullStride1_MSE"
                ]
            )
            / comparison[
                "OldScreening_MSE"
            ]
        )

        display(comparison)

        comparison.to_csv(
            RESULT_ROOT
            / "comparison_vs_exp13.csv",
            index=False,
        )


In [ ]:

print("Result root:", RESULT_ROOT)

for p in sorted(
    RESULT_ROOT.rglob("*")
):
    if p.is_file():
        print(
            p.relative_to(
                RESULT_ROOT
            )
        )
